# <span style="color:green">**Output Parsers**</span>

##### <span style="color:red">**Nota:** Este es un módulo "ipynb" creado por el **[Ing. Kevin Inofuente Colque](https://www.linkedin.com/in/kevin-inofuente-colque/)** de <span style="color:orange">**DataPath**</span> . Con mucho aprecio para mis colegas AI Enginners en toda Latinoamérica.</span> 

##### <span style="color:red">**Nota 2:** Algunos términos pueden estar en portuñol</span>

## ¿Qué es un Output Parser?

Un **Output Parser** convierte la respuesta de texto del LLM en datos estructurados (diccionarios, listas) que puedes usar en tu código.

```
Sin parser:  LLM → "texto"     → ❌ No puedes hacer datos['campo']
Con parser:  LLM → {"campo": "valor"} → ✅ Puedes hacer datos['campo']
```

## Ejemplo Simple

Queremos extraer información de una persona a partir de un texto:

**Texto:** "María García tiene 28 años y es ingeniera en Madrid."

**Resultado esperado:**
```json
{
  "nombre": "María García",
  "edad": 28,
  "profesion": "ingeniera",
  "ciudad": "Madrid"
}
```



## <span style="color:orange">**El Problema: Sin Output Parser**</span>

Veamos qué pasa cuando pedimos JSON al modelo sin usar un parser:

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

# Inicializar modelo
chat_model = init_chat_model("gpt-4.1")

# Texto simple
texto = "María García tiene 28 años y es ingeniera en Madrid."

# Prompt pidiendo JSON
prompt = ChatPromptTemplate.from_template("""
Extrae nombre, edad, profesion y ciudad del texto.
Texto: {texto}
Responde SOLO en JSON.
""")

# Invocar el modelo
respuesta = chat_model.invoke(prompt.format_messages(texto=texto))
print(respuesta.content)

{
  "nombre": "María García",
  "edad": 28,
  "profesion": "ingeniera",
  "ciudad": "Madrid"
}


In [2]:
# ❌ EL PROBLEMA: respuesta.content es un STRING, no un diccionario
print(f"Tipo: {type(respuesta.content)}")

# Esto NO funciona - da error
try:
    print(respuesta.content['nombre'])
except TypeError as e:
    print(f"❌ Error: {e}")

Tipo: <class 'str'>
❌ Error: string indices must be integers, not 'str'


## <span style="color:orange">**Nota: La forma antigua (JsonOutputParser + LCEL)**</span>

Antes de LangChain v1.0, la forma común de estructurar respuestas era con un **Output Parser** dentro de una cadena LCEL:

```
prompt | model | parser
```

Por ejemplo, `JsonOutputParser` recibía un schema Pydantic, inyectaba instrucciones de formato en el prompt, y parseaba el string de respuesta a un diccionario.

⚠️ Esta forma **aún funciona en v1.0**, pero **ya no es la recomendada** porque depende de que el LLM siga el formato pedido en el prompt y puede fallar.

A continuación veremos la **forma moderna y más confiable**: `with_structured_output()`.

## <span style="color:orange">**La Solución Moderna: `with_structured_output()` (LangChain v1.0)**</span>

En LangChain v1.0 todos los `ChatModel` exponen el método `.with_structured_output(schema)`, que devuelve un modelo "envuelto" que **siempre** responde respetando el schema indicado.

**¿Cómo lo logra?** Por debajo usa las capacidades nativas del proveedor (function calling / JSON mode), que **obligan** al modelo a producir una salida válida según el schema. Esto es mucho más confiable que pedirle por texto que devuelva JSON.

**Ventajas frente a `JsonOutputParser`:**

- ✅ **No necesitas LCEL** ni format_instructions en el prompt.
- ✅ **Devuelve directamente una instancia tipada** de Pydantic (no un dict).
- ✅ **Mucho más confiable**: el modelo no puede "olvidarse" del formato.
- ✅ **Menos código**: 1 línea en lugar de 5.

**Pasos:**

1. Definir un schema con Pydantic.
2. Llamar `chat.with_structured_output(MiSchema)`.
3. Invocar el modelo con el texto.

In [3]:
from pydantic import BaseModel, Field

# Paso 1: Definir la estructura de datos que queremos
class Persona(BaseModel):
    nombre: str = Field(description="Nombre completo")
    edad: int = Field(description="Edad en años")
    profesion: str = Field(description="Profesión o trabajo")
    ciudad: str = Field(description="Ciudad donde vive")

# Paso 2: Envolver el modelo con el schema (¡una sola línea!)
chat_model_estructurado = chat_model.with_structured_output(Persona)

print("✅ Modelo envuelto con schema Persona")

✅ Modelo envuelto con schema Persona


In [4]:
# Paso 3: Invocar directamente con el texto (sin prompt template ni format_instructions)
resultado = chat_model_estructurado.invoke(
    f"Extrae la información de la persona del siguiente texto: {texto}"
)

print(f"Tipo del resultado: {type(resultado)}")
print(f"Es instancia de Persona: {isinstance(resultado, Persona)}")
print(f"\nResultado: {resultado}")

Tipo del resultado: <class '__main__.Persona'>
Es instancia de Persona: True

Resultado: nombre='María García' edad=28 profesion='Ingeniera' ciudad='Madrid'


In [5]:
# ✅ Como es una instancia de Pydantic, accedemos a los campos como atributos (con autocompletado)
print(f"Nombre:    {resultado.nombre}")
print(f"Edad:      {resultado.edad}")
print(f"Profesión: {resultado.profesion}")
print(f"Ciudad:    {resultado.ciudad}")

Nombre:    María García
Edad:      28
Profesión: Ingeniera
Ciudad:    Madrid


In [6]:
# Probemos con otro texto - el mismo modelo envuelto funciona
otro_texto = "Carlos López, de 35 años, es médico en Barcelona."

resultado2 = chat_model_estructurado.invoke(
    f"Extrae la información de la persona del siguiente texto: {otro_texto}"
)

print(f"Nombre: {resultado2.nombre}, Edad: {resultado2.edad}, Profesión: {resultado2.profesion}, Ciudad: {resultado2.ciudad}")

Nombre: Carlos López, Edad: 35, Profesión: médico, Ciudad: Barcelona
